In [1]:
!pip install pysbd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 2.4 MB/s eta 0:00:00


In [2]:
import pysbd
import math
import os
import logging
import torch
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (AutoTokenizer, AutoModelForMaskedLM,
                          DataCollatorForLanguageModeling,
                          TrainingArguments, Trainer, set_seed)
 
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Silence the benign ">512 tokens" warning: long docs are chopped into blocks
# below, so nothing over-length ever reaches the model.
logging.getLogger("transformers.tokenization_utils_base").setLevel(logging.ERROR)

CONFIGURATION

In [3]:
Checkpoint = "roberta-base"
Out_directory = "/kaggle/working/dapt/roberta"
Block_size = 512
MLM_PROB = 0.15
Seed = 42

Loading in JSONL files

In [4]:
Data_file_path = "/kaggle/input/datasets/lorenzouberti/unlabbled-fomc-data/*.jsonl"
raw_unlablled_jsonl = load_dataset("json", data_files={"train": Data_file_path})["train"]
raw_unlablled_jsonl = raw_unlablled_jsonl.filter(lambda x: isinstance(x["text"], str) and len(x["text"].strip()) > 0)
raw_unlablled_jsonl = raw_unlablled_jsonl.train_test_split(test_size=0.01, seed=Seed) 
raw = raw_unlablled_jsonl.copy()

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/18819 [00:00<?, ? examples/s]

In [5]:
from datasets import Dataset, DatasetDict, concatenate_datasets

print(type(raw))   # plain dict = the culprit

if isinstance(raw, dict) and not isinstance(raw, DatasetDict):
    # hand-built dict (e.g. one entry per file): merge the values into one Dataset
    parts = [d["train"] if isinstance(d, DatasetDict) else d for d in raw.values()]
    raw = concatenate_datasets(parts)

if isinstance(raw, Dataset):
    # single merged Dataset: create the 1% perplexity holdout
    raw = raw.train_test_split(test_size=0.01, seed=Seed)

print(raw)   # should show DatasetDict({'train': ..., 'test': ...})

<class 'dict'>
DatasetDict({
    train: Dataset({
        features: ['text', 'source', 'path', 'chunk', 'n_tokens'],
        num_rows: 18630
    })
    test: Dataset({
        features: ['text', 'source', 'path', 'chunk', 'n_tokens'],
        num_rows: 189
    })
})


Tokenize

In [6]:
def split_docs(batch):
    seg = pysbd.Segmenter(language="en", clean=False)
    out = []
    for doc in batch["text"]:
        sents = []
        for para in doc.split("\n\n"):   # paragraph pre-split (no-op if none present)
            para = para.strip()
            if para:
                sents.extend(s.strip() for s in seg.segment(para) if s.strip())
        out.append(sents)
    return {"sentences": out}
 
sent_ds = raw.map(split_docs, batched=True,
                  remove_columns=raw["train"].column_names,  # drops text AND any metadata
                  num_proc=4)
 
# Persist so you never pay the pysbd cost again:
sent_ds.save_to_disk("/kaggle/working/sentences")

Map (num_proc=4):   0%|          | 0/18630 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/189 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/18630 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/189 [00:00<?, ? examples/s]

In [7]:
tokenizer = AutoTokenizer.from_pretrained(Checkpoint)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
CLS_ID = tokenizer.cls_token_id
SEP_ID = tokenizer.sep_token_id
assert CLS_ID is not None and SEP_ID is not None, "checkpoint lacks CLS/SEP tokens"
BUDGET = Block_size - 2                     # room for the two special tokens
 
def pack(batch):
    docs = batch["sentences"]
    flat = [s for doc in docs for s in doc]  # batch-tokenize everything in one call
    enc = tokenizer(flat, add_special_tokens=False)["input_ids"]
 
    input_ids, special_masks = [], []
 
    def emit(cur):
        input_ids.append([CLS_ID] + cur + [SEP_ID])
        special_masks.append([1] + [0] * len(cur) + [1])   # 1 = never MLM-mask
 
    i = 0
    for doc in docs:
        cur = []
        for _ in doc:
            ids = enc[i][:BUDGET]   # hard-truncate rare monster "sentences" (tables etc.)
            i += 1
            if not ids:
                continue
            if len(cur) + len(ids) > BUDGET and cur:
                emit(cur)
                cur = []
            cur.extend(ids)
        if cur:                     # flush at doc end -> blocks never cross documents
            emit(cur)
    return {"input_ids": input_ids, "special_tokens_mask": special_masks}
 
lm_ds = sent_ds.map(pack, batched=True, remove_columns=["sentences"], num_proc=4)
print(lm_ds)   # ~30M tokens / ~512 per block ≈ 58k blocks

Map (num_proc=4):   0%|          | 0/18630 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/189 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'special_tokens_mask'],
        num_rows: 69324
    })
    test: Dataset({
        features: ['input_ids', 'special_tokens_mask'],
        num_rows: 708
    })
})


In [9]:
model = AutoModelForMaskedLM.from_pretrained(Checkpoint)
 
collator = DataCollatorForLanguageModeling(tokenizer, mlm=True,
                                           mlm_probability=MLM_PROB,
                                           pad_to_multiple_of=8)  # tensor-core friendly for fp16
 
args = TrainingArguments(
    output_dir=Out_directory,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=1,                # keeps you under Kaggle's 20GB working-dir cap
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    learning_rate=5e-5,
    per_device_train_batch_size=8,     # 16 OOMs on the 15GB T4: MLM logits are batch x 512 x vocab
    per_device_eval_batch_size=8,      # eval logits spike the same way
    gradient_accumulation_steps=8,     # effective batch stays 64 -> same optimization as before
    # gradient_checkpointing=True,     # next lever if it STILL OOMs (~30-40% slower)
    num_train_epochs=5,
    warmup_ratio=0.06,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
    seed=Seed,
)
 
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=lm_ds["train"],
    eval_dataset=lm_ds["test"],
    processing_class=tokenizer,
    data_collator=collator,
)
 
# %%
trainer.train()
 
eval_out = trainer.evaluate()
print(f"eval loss {eval_out['eval_loss']:.4f} | perplexity {math.exp(eval_out['eval_loss']):.2f}")
 
# %%
final_dir = f"{Out_directory}/final"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss
250,26.775884,3.084293
500,25.179243,2.895684
750,24.092886,2.776618
1000,23.357942,2.622470
1250,22.555334,2.613633
1500,22.143245,2.565621
1750,21.900740,2.478838
2000,21.552969,2.444494
2250,21.512505,2.468379
2500,21.281194,2.464071


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

eval loss 2.4427 | perplexity 11.50


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/dapt/roberta/final/tokenizer_config.json',
 '/kaggle/working/dapt/roberta/final/tokenizer.json')